In [1]:
from __future__ import annotations

import numpy
import random
import pandas
import time
import os
from typing import Any
from deap import creator
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import matthews_corrcoef
from scipy.stats import pointbiserialr

from training_config import TrainingConfig
from training_utils import best_sign_consistency_index, knee_point_index
from multi_objective_training import MultiObjectiveTraining
from single_objective_training import SingleObjectiveTraining
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression as _SFSLogReg
from evaluation_utils import (
    get_continuous_columns, get_dummy_columns,
    build_model_package, evaluate_and_save,
    compute_stability_matrix, plot_stability_heatmap, plot_feature_frequency,
    compute_vif_across_seeds, plot_vif_boxplot,
)
from evaluation_utils import apply_proportional_noise, apply_dummy_noise, evaluate_model
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression as _CmpLogReg
import matplotlib.pyplot as plt

In [2]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    numpy.random.seed(seed)

In [3]:
#CSV_TRAIN_PATH: str = "readmit/readmit_130_hospitals_preprocessed_train_data.csv"
#CSV_TEST_PATH: str = "readmit/readmit_130_hospitals_preprocessed_test_data.csv"
#TARGET_COLUMN: str = "target_readmitted"
CSV_TRAIN_PATH: str = "arrhythmia/arrhythmia_preprocessed_train_data.csv"
CSV_TEST_PATH: str = "arrhythmia/arrhythmia_preprocessed_test_data.csv"
TARGET_COLUMN: str = "Outcome"
#CSV_TRAIN_PATH: str = "hmeq/hmeq_preprocessed_train_data.csv"
#CSV_TEST_PATH: str = "hmeq/hmeq_preprocessed_test_data.csv"
#TARGET_COLUMN: str = "BAD"
#CSV_TRAIN_PATH: str = "D:\\RoutePatch\\GitWorking\\multi-objective-regression\\test_data\\rad_fusion_train_modified.csv"
#CSV_VALIDATION_PATH: str = "D:\\RoutePatch\\GitWorking\\multi-objective-regression\\test_data\\rad_fusion_validation_modified.csv"
#CSV_TEST_PATH: str = "D:\\RoutePatch\\GitWorking\\multi-objective-regression\\test_data\\rad_fusion_test_modified.csv"
#TARGET_COLUMN: str = "label"

RESULT_PATH: str = time.strftime("%Y-%m-%d_%H-%M-%S")
RESULT_PATH_MULTI: str = f"{RESULT_PATH}\\multi"
RESULT_PATH_SINGLE: str = f"{RESULT_PATH}\\single"
RESULT_PATH_EVAL: str = f"{RESULT_PATH}\\evaluation"

USE_KNEE_POINT_SELECTION: bool = False

In [4]:
# Load train set
df_train: pandas.DataFrame = pandas.read_csv(CSV_TRAIN_PATH)

# Split data into training and validation sets
X_search_pandas: pandas.DataFrame = df_train.drop(columns=[TARGET_COLUMN])
y_search_pandas: pandas.Series = df_train[TARGET_COLUMN]

X_search: numpy.ndarray = numpy.ascontiguousarray(X_search_pandas.to_numpy(), dtype=numpy.float32)
y_search: numpy.ndarray = numpy.ascontiguousarray(y_search_pandas.to_numpy(), dtype=numpy.float32)

feature_names: list[str] = list(X_search_pandas.columns)

cv: StratifiedKFold = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [5]:
# Load test set
df_test: pandas.DataFrame = pandas.read_csv(CSV_TEST_PATH)

y_test: pandas.Series = df_test[TARGET_COLUMN]
X_test: pandas.DataFrame = df_test.drop(columns=[TARGET_COLUMN])

In [6]:
# Pre-compute folds and scaling
fold_indices: list[tuple[numpy.ndarray, numpy.ndarray]] = list(cv.split(X_search, y_search))

X_train_scaled_folds: list[numpy.ndarray] = []
X_val_scaled_folds: list[numpy.ndarray] = []
y_train_folds: list[numpy.ndarray] = []
y_val_folds: list[numpy.ndarray] = []

n_splits: int = len(fold_indices)
n_features: int = len(feature_names)
corr_matrix: numpy.ndarray = numpy.zeros((n_splits, n_features), dtype=float)
for fold_idx, (train_idx, val_idx) in enumerate(fold_indices):
    X_fold_train: numpy.ndarray = X_search[train_idx]
    X_fold_val: numpy.ndarray = X_search[val_idx]
    y_fold_train: numpy.ndarray = y_search[train_idx]
    y_fold_val: numpy.ndarray = y_search[val_idx]

    # Pre-scale (all features at once)
    scaler: StandardScaler = StandardScaler()
    X_train_scaled_folds.append(scaler.fit_transform(X_fold_train))
    X_val_scaled_folds.append(scaler.transform(X_fold_val))
    y_train_folds.append(y_fold_train)
    y_val_folds.append(y_fold_val)

    # Calculate Point-biserial correlation coefficients (target value is binary and the input variables are continuous)
    # Calculate the average correlation for each feature across all fold
    for col_idx in range(X_fold_train.shape[1]):
        feature_col: numpy.ndarray = X_fold_train[:, col_idx]
        unique_vals: int = len(numpy.unique(feature_col))

        if unique_vals <= 1:
            corr: float = 0.0
        elif unique_vals == 2:
            corr: float = matthews_corrcoef(y_fold_train, feature_col.astype(int))
        else:
            corr, _ = pointbiserialr(y_fold_train, feature_col)

        corr_matrix[fold_idx, col_idx] = corr

In [ ]:
training_results_multi: dict[int, list[list[creator.Individual]]] = {}
training_results_single: dict[int, list[creator.IndividualSingle]] = {}

# 20 seeds: enough for tight CIs on the mean and adequate power for paired
# tests; deterministic ordering keeps results comparable across runs.
seeds: list[int] = list(range(42, 62))
for s in seeds:
    set_seed(s)
    multi_objective_training: MultiObjectiveTraining = MultiObjectiveTraining(
        config=TrainingConfig(seed=s, result_directory=RESULT_PATH_MULTI),
        feature_names=feature_names,
        fold_indices=fold_indices,
        X_train_scaled_folds=X_train_scaled_folds,
        X_val_scaled_folds=X_val_scaled_folds,
        y_train_folds=y_train_folds,
        y_val_folds=y_val_folds,
        corr_matrix=corr_matrix,
    )
    pareto_front: list[creator.Individual] = multi_objective_training.run()
    training_results_multi[s] = pareto_front
    multi_objective_training.clear_cache()

    set_seed(s)
    single_objective_training: SingleObjectiveTraining = SingleObjectiveTraining(
        config=TrainingConfig(seed=s, result_directory=RESULT_PATH_SINGLE),
        feature_names=feature_names,
        fold_indices=fold_indices,
        X_train_scaled_folds=X_train_scaled_folds,
        X_val_scaled_folds=X_val_scaled_folds,
        y_train_folds=y_train_folds,
        y_val_folds=y_val_folds)
    best_indi: creator.IndividualSingle = single_objective_training.run()
    training_results_single[s] = best_indi
    single_objective_training.clear_cache()

Generation 1 done | Pareto size: 7
Generation 2 done | Pareto size: 9
Generation 3 done | Pareto size: 8
Generation 4 done | Pareto size: 6
Generation 5 done | Pareto size: 8
Generation 6 done | Pareto size: 5
Generation 7 done | Pareto size: 9
Generation 8 done | Pareto size: 11
Generation 9 done | Pareto size: 8
Generation 10 done | Pareto size: 15
Generation 11 done | Pareto size: 12
Generation 12 done | Pareto size: 7
Generation 13 done | Pareto size: 8
Generation 14 done | Pareto size: 8
Generation 15 done | Pareto size: 8
Generation 16 done | Pareto size: 9


## Evaluation: Single vs Multi-Objective Robustness Comparison

For each seed we:
1. Retrain a final LogisticRegression on the **full training set** using features selected by the GA.
   - Multi-objective: the Pareto individual with the **highest sign-consistency** score.
   - Single-objective: the best individual from the Hall of Fame.
2. Evaluate on clean data and under increasing noise levels on **both test and validation (training) sets**:
   - **Gaussian noise + mean shift** on continuous features
   - **Random corruption noise** on dummy (binary) features
3. Save CSV results and comparison plots to the result directory (separate files per dataset).

In [ ]:
# Detect column types automatically from the training DataFrame
X_train_df: pandas.DataFrame = df_train.drop(columns=[TARGET_COLUMN])
y_train_series: pandas.Series = df_train[TARGET_COLUMN]

continuous_cols: list[str] = get_continuous_columns(X_train_df)
dummy_cols: list[str] = get_dummy_columns(X_train_df)

# Training-set std for proportional noise scaling
train_std: pandas.Series = X_train_df.std()

print(f"Continuous features: {len(continuous_cols)}")
print(f"Dummy features:     {len(dummy_cols)}")

In [ ]:
# Multi-objective: pick the Pareto individual with the highest sign-consistency
evaluation_results_test: dict[int, pandas.DataFrame] = {}

for s in seeds:
    set_seed(s)

    # --- Multi-objective: select max sign-consistency or knee point individual from Pareto front ---
    pareto_front: list[creator.Individual] = training_results_multi[s]
    best_multi_ind: creator.Individual = pareto_front[knee_point_index(pareto_front) if USE_KNEE_POINT_SELECTION else best_sign_consistency_index(pareto_front)]
    print(f"Seed {s} | Selected multi-objective individual: "
          f"AUC={best_multi_ind.fitness.values[0]:.4f}, "
          f"SignConsistency={best_multi_ind.fitness.values[1]:.4f}, "
          f"Features={sum(best_multi_ind)}")

    # --- Build final model packages on full training set ---
    model_pkg_multi = build_model_package(
        best_multi_ind, feature_names, X_train_df, y_train_series, seed=s)
    model_pkg_single = build_model_package(
        training_results_single[s], feature_names, X_train_df, y_train_series, seed=s)

    # --- Run noise evaluation on TEST set and save plots/CSV ---
    eval_df = evaluate_and_save(
        seed=s,
        model_pkg_multi=model_pkg_multi,
        model_pkg_single=model_pkg_single,
        X_eval=X_test,
        y_eval=y_test,
        train_std=train_std,
        continuous_cols=continuous_cols,
        dummy_cols=dummy_cols,
        result_directory=RESULT_PATH_EVAL,
        use_roc_auc=True,
        dataset_label="Test",
    )
    evaluation_results_test[s] = eval_df

print("\nTest set evaluation complete. Results saved to:", RESULT_PATH_EVAL)

In [ ]:
# Noise evaluation on VALIDATION (training) set
evaluation_results_val: dict[int, pandas.DataFrame] = {}

for s in seeds:
    set_seed(s)

    # --- Multi-objective: select max sign-consistency or knee point individual from Pareto front ---
    pareto_front: list[creator.Individual] = training_results_multi[s]
    best_multi_ind: creator.Individual = pareto_front[knee_point_index(pareto_front) if USE_KNEE_POINT_SELECTION else best_sign_consistency_index(pareto_front)]

    # --- Build final model packages on full training set ---
    model_pkg_multi = build_model_package(
        best_multi_ind, feature_names, X_train_df, y_train_series, seed=s)
    model_pkg_single = build_model_package(
        training_results_single[s], feature_names, X_train_df, y_train_series, seed=s)

    # --- Run noise evaluation on VALIDATION (training) set and save plots/CSV ---
    eval_df = evaluate_and_save(
        seed=s,
        model_pkg_multi=model_pkg_multi,
        model_pkg_single=model_pkg_single,
        X_eval=X_train_df,
        y_eval=y_train_series,
        train_std=train_std,
        continuous_cols=continuous_cols,
        dummy_cols=dummy_cols,
        result_directory=RESULT_PATH_EVAL,
        use_roc_auc=True,
        dataset_label="Validation",
    )
    evaluation_results_val[s] = eval_df

print("\nValidation set evaluation complete. Results saved to:", RESULT_PATH_EVAL)

## Feature Selection Stability (Jaccard Similarity)

Compare how consistently each method selects the same features across different random seeds.
- **Jaccard similarity** = |intersection| / |union| between two feature sets (1.0 = identical, 0.0 = no overlap).
- A **pairwise heatmap** shows seed-to-seed stability for multi-objective vs single-objective.
- A **feature frequency bar chart** shows which features are consistently selected across seeds.

In [ ]:
# Collect the best individual per seed for both methods
best_multi_by_seed: dict[int, list[int]] = {}
best_single_by_seed: dict[int, list[int]] = {}

for s in seeds:
    # Multi-objective: max sign-consistency or knee point (same selection as evaluation)
    pareto_front: list[creator.Individual] = training_results_multi[s]
    best_multi_ind: creator.Individual = pareto_front[knee_point_index(pareto_front) if USE_KNEE_POINT_SELECTION else best_sign_consistency_index(pareto_front)]
    best_multi_by_seed[s]: list[int] = list(best_multi_ind)

    # Single-objective: best individual from HoF
    best_single_by_seed[s]: list[int] = list(training_results_single[s])

# Compute pairwise Jaccard stability matrices
seeds_multi, matrix_multi = compute_stability_matrix(best_multi_by_seed, feature_names)
seeds_single, matrix_single = compute_stability_matrix(best_single_by_seed, feature_names)

# Print summary
print("=== Feature Selection Stability (Jaccard Similarity) ===\n")
print("Multi-Objective (SiCo-MOGA) pairwise Jaccard:")
for i, si in enumerate(seeds_multi):
    for j, sj in enumerate(seeds_multi):
        if j > i:
            print(f"  Seed {si} vs Seed {sj}: {matrix_multi[i, j]:.3f}")

print("\nSingle-Objective (AUC-only GA) pairwise Jaccard:")
for i, si in enumerate(seeds_single):
    for j, sj in enumerate(seeds_single):
        if j > i:
            print(f"  Seed {si} vs Seed {sj}: {matrix_single[i, j]:.3f}")

# Save plots
stability_dir: str = os.path.join(RESULT_PATH_EVAL, "stability")

plot_stability_heatmap(
    seeds_multi, matrix_multi,
    seeds_single, matrix_single,
    os.path.join(stability_dir, "jaccard_heatmap.png"))

plot_feature_frequency(
    best_multi_by_seed, feature_names,
    "Multi-Objective (SiCo-MOGA)",
    os.path.join(stability_dir, "feature_frequency_multi.png"))

plot_feature_frequency(
    best_single_by_seed, feature_names,
    "Single-Objective (AUC-only GA)",
    os.path.join(stability_dir, "feature_frequency_single.png"))

print(f"\nStability plots saved to: {stability_dir}")

## Multicollinearity Analysis: Variance Inflation Factor (VIF)

VIF measures how much the variance of a regression coefficient is inflated due to multicollinearity.
- **VIF = 1**: no correlation with other predictors
- **VIF = 5**: moderate multicollinearity
- **VIF > 10**: high multicollinearity (coefficient estimates become unreliable)

We compare the VIF distributions of features selected by SOGA (Single-Objective) vs MOGA (Multi-Objective) across all seeds.

In [ ]:
# Compute VIF for features selected by each method across seeds
vif_multi: pandas.DataFrame = compute_vif_across_seeds(best_multi_by_seed, feature_names, X_train_df)
vif_single: pandas.DataFrame = compute_vif_across_seeds(best_single_by_seed, feature_names, X_train_df)

print("=== Variance Inflation Factor (VIF) Summary ===\n")
print("Multi-Objective (SiCo-MOGA):")
print(f"  Features per seed: {vif_multi.groupby('seed')['feature'].count().values}")
print(f"  Median VIF: {vif_multi['vif'].median():.2f}")
print(f"  Mean VIF:   {vif_multi['vif'].mean():.2f}")
print(f"  Max VIF:    {vif_multi['vif'].max():.2f}")
print(f"  Features with VIF > 5: {(vif_multi['vif'] > 5).sum()} / {len(vif_multi)}")

print("\nSingle-Objective (AUC-only GA):")
print(f"  Features per seed: {vif_single.groupby('seed')['feature'].count().values}")
print(f"  Median VIF: {vif_single['vif'].median():.2f}")
print(f"  Mean VIF:   {vif_single['vif'].mean():.2f}")
print(f"  Max VIF:    {vif_single['vif'].max():.2f}")
print(f"  Features with VIF > 5: {(vif_single['vif'] > 5).sum()} / {len(vif_single)}")

# Save VIF plot
vif_dir: str = os.path.join(RESULT_PATH_EVAL, "vif")
plot_vif_boxplot(vif_multi, vif_single, os.path.join(vif_dir, "vif_comparison.png"))

# Save VIF data as CSV
vif_multi.to_csv(os.path.join(vif_dir, "vif_multi.csv"), index=False)
vif_single.to_csv(os.path.join(vif_dir, "vif_single.csv"), index=False)

print(f"\nVIF plots and data saved to: {vif_dir}")

## Feature Set Comparison: Multi-Objective vs Single-Objective

For each seed, compare the features selected by SiCo-MOGA vs. SOGA:
- **Only in Multi**: features that the multi-objective GA selected but single-objective did not
- **Only in Single**: features that the single-objective GA selected but multi-objective did not
- **Common**: features selected by both

This highlights which features the sign-consistency penalty specifically filters out (only-in-single) or adds back (only-in-multi). The results are printed per seed and saved as a CSV.

In [ ]:
# Feature set comparison: multi vs single objective
from evaluation_utils import ensure_directory as _ensure_directory

comparison_rows: list[dict] = []

print("=== Feature Set Comparison: Multi-Objective vs Single-Objective ===\n")
for s in seeds:
    multi_bits: list[int] = best_multi_by_seed[s]
    single_bits: list[int] = best_single_by_seed[s]

    multi_set: set[str] = {f for f, b in zip(feature_names, multi_bits) if b == 1}
    single_set: set[str] = {f for f, b in zip(feature_names, single_bits) if b == 1}

    only_in_multi: list[str] = sorted(multi_set - single_set)
    only_in_single: list[str] = sorted(single_set - multi_set)
    common: list[str] = sorted(multi_set & single_set)

    print(f"--- Seed {s} ---")
    print(f"  Multi features: {len(multi_set)} | Single features: {len(single_set)} | Common: {len(common)}")
    print(f"  Only in Multi  ({len(only_in_multi)}):")
    for f in only_in_multi:
        print(f"    + {f}")
    print(f"  Only in Single ({len(only_in_single)}):")
    for f in only_in_single:
        print(f"    - {f}")
    print()

    for f in only_in_multi:
        comparison_rows.append({"seed": s, "feature": f, "membership": "only_in_multi"})
    for f in only_in_single:
        comparison_rows.append({"seed": s, "feature": f, "membership": "only_in_single"})
    for f in common:
        comparison_rows.append({"seed": s, "feature": f, "membership": "common"})

comparison_df: pandas.DataFrame = pandas.DataFrame(comparison_rows)

comparison_dir: str = os.path.join(RESULT_PATH_EVAL, "feature_comparison")
_ensure_directory(comparison_dir)
comparison_df.to_csv(os.path.join(comparison_dir, "feature_set_comparison.csv"), index=False)

print(f"Feature set comparison CSV saved to: {comparison_dir}")

## Sign Inconsistency in Single-Objective Model

For each feature selected by the Single-Objective GA, compare:
- **Marginal correlation sign** with the target (point-biserial for continuous, Matthews for binary), computed on the full training set
- **Logistic regression coefficient sign**, from the final model retrained on the full training set

Features where these signs **disagree** are sign-inconsistent: the feature's individual association with the target is opposite to how the model uses it (a hallmark of suppressor effects and multicollinearity). The sign-consistency objective in SiCo-MOGA is designed to avoid exactly these cases.

In [ ]:
# Sign inconsistency in Single-Objective model: correlation sign vs coefficient sign

# Pre-compute marginal correlations with target on the FULL training set
# Cast to int for matthews_corrcoef so sklearn's type_of_target detects it as
# binary rather than continuous (ValueError otherwise when the column dtype is float).
y_full: numpy.ndarray = y_train_series.to_numpy().astype(int)

full_train_corr: dict[str, float] = {}
for col in feature_names:
    feature_col: numpy.ndarray = X_train_df[col].to_numpy()
    unique_vals: int = len(numpy.unique(feature_col))
    if unique_vals <= 1:
        full_train_corr[col] = 0.0
    elif unique_vals == 2:
        full_train_corr[col] = float(matthews_corrcoef(y_full, feature_col.astype(int)))
    else:
        c, _ = pointbiserialr(y_full, feature_col)
        full_train_corr[col] = float(c)


def _sign(x: float, atol: float = 1e-12) -> int:
    """Return -1/0/+1 with a tolerance around zero."""
    if numpy.isclose(x, 0.0, atol=atol):
        return 0
    return 1 if x > 0 else -1


inconsistency_rows: list[dict] = []

print("=== Sign Inconsistency in Single-Objective Selected Features ===\n")
for s in seeds:
    set_seed(s)
    model_pkg = build_model_package(
        training_results_single[s], feature_names, X_train_df, y_train_series, seed=s)

    selected: list[str] = model_pkg["features"]
    coefs: numpy.ndarray = model_pkg["model"].coef_[0]

    inconsistent: list[dict] = []
    for feat, coef in zip(selected, coefs):
        corr: float = full_train_corr[feat]
        corr_sign: int = _sign(corr)
        coef_sign: int = _sign(float(coef))
        mismatch: bool = (corr_sign != 0 and coef_sign != 0 and corr_sign != coef_sign)
        if mismatch:
            inconsistent.append({
                "seed": s,
                "feature": feat,
                "correlation": corr,
                "coefficient": float(coef),
                "corr_sign": corr_sign,
                "coef_sign": coef_sign,
            })

    print(f"--- Seed {s} ---")
    print(f"  Single-objective selected features: {len(selected)}")
    print(f"  Sign-inconsistent features:         {len(inconsistent)} "
          f"({100.0 * len(inconsistent) / max(len(selected), 1):.1f}%)")
    if inconsistent:
        print(f"  {'Feature':<40} {'Correlation':>12} {'Coefficient':>14}")
        for row in sorted(inconsistent, key=lambda r: abs(r["coefficient"]), reverse=True):
            print(f"    {row['feature']:<38} {row['correlation']:>12.4f} {row['coefficient']:>14.4f}")
    print()

    inconsistency_rows.extend(inconsistent)

inconsistency_df: pandas.DataFrame = pandas.DataFrame(inconsistency_rows)

inconsistency_dir: str = os.path.join(RESULT_PATH_EVAL, "feature_comparison")
_ensure_directory(inconsistency_dir)
inconsistency_df.to_csv(os.path.join(inconsistency_dir, "single_sign_inconsistency.csv"), index=False)

print(f"Sign inconsistency CSV saved to: {inconsistency_dir}")

## Baseline: Logistic Regression with ALL features (no feature selection)

Train a logistic regression using **every** available feature (no GA, no stepwise search) and report clean AUC/AP on the test set per seed. Serves as the trivial "no selection" reference point against which both the GA-based methods and the forward-stepwise baseline must justify themselves.

In [ ]:
# Baseline 1: Logistic Regression using ALL features (no feature selection)
baseline_all_rows: list[dict] = []
all_features_ind: list[int] = [1] * len(feature_names)

print("=== Baseline: Logistic Regression with ALL features ===\n")
for s in seeds:
    set_seed(s)

    model_pkg_all: dict[str, Any] = build_model_package(
        all_features_ind, feature_names, X_train_df, y_train_series, seed=s)

    X_test_scaled_all: numpy.ndarray = model_pkg_all["scaler"].transform(
        X_test[model_pkg_all["features"]].to_numpy())
    y_prob_all: numpy.ndarray = model_pkg_all["model"].predict_proba(X_test_scaled_all)[:, 1]

    test_auc_all: float = float(roc_auc_score(y_test, y_prob_all))

    baseline_all_rows.append({
        "seed": s,
        "n_features": len(model_pkg_all["features"]),
        "test_auc": test_auc_all,
    })
    print(f"  Seed {s} | Features: {len(model_pkg_all['features']):4d} "
          f"| Test AUC: {test_auc_all:.4f}")

baseline_all_df: pandas.DataFrame = pandas.DataFrame(baseline_all_rows)
print(f"\nMean test AUC: {baseline_all_df['test_auc'].mean():.4f} "
      f"(+/- {baseline_all_df['test_auc'].std():.4f})")

baseline_all_dir: str = os.path.join(RESULT_PATH_EVAL, "baseline_all_features")
_ensure_directory(baseline_all_dir)
baseline_all_df.to_csv(
    os.path.join(baseline_all_dir, "baseline_all_features.csv"), index=False)
print(f"\nBaseline (all features) results saved to: {baseline_all_dir}")

## Baseline: Logistic Regression with Forward Stepwise Feature Selection

Use scikit-learn's `SequentialFeatureSelector` (`direction="forward"`, 3-fold stratified CV on ROC AUC) to greedily add features while the cross-validated AUC keeps improving by more than `tol=1e-3`. Refit the final logistic regression on the selected subset via `build_model_package`, then report clean AUC/AP on the test set. This is the classical stepwise baseline the GA methods must justify improvement over.

In [ ]:
# Baseline 2: Logistic Regression with forward stepwise feature selection
baseline_fwd_rows: list[dict] = []

print("=== Baseline: Logistic Regression with Forward Stepwise Feature Selection ===\n")
for s in seeds:
    set_seed(s)

    # SFS needs scaled features; it refits LR internally at each step
    sfs_scaler: StandardScaler = StandardScaler()
    X_train_sfs: numpy.ndarray = sfs_scaler.fit_transform(X_train_df.to_numpy())

    sfs_cv: StratifiedKFold = StratifiedKFold(n_splits=3, shuffle=True, random_state=s)
    base_lr: _SFSLogReg = _SFSLogReg(
        penalty="l2", solver="lbfgs", max_iter=1000, random_state=s)
    sfs: SequentialFeatureSelector = SequentialFeatureSelector(
        base_lr,
        n_features_to_select="auto",
        tol=1e-3,
        direction="forward",
        scoring="roc_auc",
        cv=sfs_cv,
        n_jobs=-1,
    )
    sfs.fit(X_train_sfs, y_train_series)

    selected_mask: numpy.ndarray = sfs.get_support()
    fwd_individual: list[int] = [1 if m else 0 for m in selected_mask]

    # Re-fit the final LR on the selected subset through the standard helper
    model_pkg_fwd = build_model_package(
        fwd_individual, feature_names, X_train_df, y_train_series, seed=s)

    X_test_scaled_fwd: numpy.ndarray = model_pkg_fwd["scaler"].transform(
        X_test[model_pkg_fwd["features"]].to_numpy())
    y_prob_fwd: numpy.ndarray = model_pkg_fwd["model"].predict_proba(X_test_scaled_fwd)[:, 1]

    test_auc_fwd: float = float(roc_auc_score(y_test, y_prob_fwd))

    baseline_fwd_rows.append({
        "seed": s,
        "n_features": len(model_pkg_fwd["features"]),
        "test_auc": test_auc_fwd,
        "features": ",".join(model_pkg_fwd["features"]),
    })
    print(f"  Seed {s} | Features: {len(model_pkg_fwd['features']):4d} "
          f"| Test AUC: {test_auc_fwd:.4f}")

baseline_fwd_df: pandas.DataFrame = pandas.DataFrame(baseline_fwd_rows)
print(f"\nMean features: {baseline_fwd_df['n_features'].mean():.1f} "
      f"(+/- {baseline_fwd_df['n_features'].std():.1f})")
print(f"Mean test AUC: {baseline_fwd_df['test_auc'].mean():.4f} "
      f"(+/- {baseline_fwd_df['test_auc'].std():.4f})")

baseline_fwd_dir: str = os.path.join(RESULT_PATH_EVAL, "baseline_forward_stepwise")
_ensure_directory(baseline_fwd_dir)
baseline_fwd_df.to_csv(
    os.path.join(baseline_fwd_dir, "baseline_forward_stepwise.csv"), index=False)
print(f"\nBaseline (forward stepwise) results saved to: {baseline_fwd_dir}")

## All-Models Noise Robustness Comparison (averaged across seeds)

Evaluate all four models on the **test set** under two noise sweeps:
- **Gaussian noise** on continuous features (zero mean-shift), noise level = 0.0 .. 1.0 in 0.1 steps.
- **Random corruption** on binary (dummy) features, corruption fraction = 0.0 .. 1.0 in 0.1 steps.

For each seed the same four models are evaluated at every noise level, and the mean AUC across seeds is plotted with a ±1 std shaded band. This gives a like-for-like view of robustness for:
- **Multi-Objective (SiCo-MOGA)** — Pareto individual with max sign-consistency or knee point
- **Single-Objective (AUC-only GA)** — best HoF individual
- **All features (no selection)** — logistic regression on every input
- **Forward stepwise selection** — `SequentialFeatureSelector` (forward, CV=3, ROC-AUC, tol=1e-3)

The two output plots match the style of the per-seed `gaussian_noise_comparison_test.png` and `dummy_flip_comparison_test.png` files, but with four curves and seed-aggregated statistics.

In [ ]:
# All-models noise robustness comparison (averaged across seeds)
noise_levels: numpy.ndarray = numpy.arange(0.0, 1.1, 0.1)
flip_levels: numpy.ndarray = numpy.arange(0.0, 1.05, 0.1)

gauss_rows: list[dict] = []
dummy_rows: list[dict] = []

all_features_ind: list[int] = [1] * len(feature_names)

print("Running noise sweeps for all 4 models across all seeds...")
for s in seeds:
    set_seed(s)

    # --- Build all 4 model packages on the same full training set ---
    pareto_front: list[creator.Individual] = training_results_multi[s]
    best_multi_ind: creator.Individual = pareto_front[knee_point_index(pareto_front) if USE_KNEE_POINT_SELECTION else best_sign_consistency_index(pareto_front)]

    pkg_multi: dict[str, Any] = build_model_package(
        best_multi_ind, feature_names, X_train_df, y_train_series, seed=s)
    pkg_single: dict[str, Any] = build_model_package(
        training_results_single[s], feature_names, X_train_df, y_train_series, seed=s)
    pkg_all: dict[str, Any] = build_model_package(
        all_features_ind, feature_names, X_train_df, y_train_series, seed=s)

    # Forward stepwise: re-run SFS per seed (CV shuffle depends on seed)
    sfs_scaler: StandardScaler = StandardScaler()
    X_train_sfs: numpy.ndarray = sfs_scaler.fit_transform(X_train_df.to_numpy())
    sfs_cv: StratifiedKFold = StratifiedKFold(n_splits=3, shuffle=True, random_state=s)
    base_lr: _CmpLogReg = _CmpLogReg(
        penalty="l2", solver="lbfgs", max_iter=1000, random_state=s)
    sfs: SequentialFeatureSelector = SequentialFeatureSelector(
        base_lr,
        n_features_to_select="auto",
        tol=1e-3,
        direction="forward",
        scoring="roc_auc",
        cv=sfs_cv,
        n_jobs=-1,
    )
    sfs.fit(X_train_sfs, y_train_series)
    fwd_individual: list[int] = [1 if m else 0 for m in sfs.get_support()]
    pkg_fwd = build_model_package(
        fwd_individual, feature_names, X_train_df, y_train_series, seed=s)

    models: dict[str, dict] = {
        "multi":   pkg_multi,
        "single":  pkg_single,
        "all":     pkg_all,
        "forward": pkg_fwd,
    }

    # --- Gaussian noise sweep (zero mean-shift) ---
    for noise in noise_levels:
        nv: float = round(float(noise), 2)
        X_noisy: pandas.DataFrame = apply_proportional_noise(
            X_test, train_std, nv, 0.0, continuous_cols)
        row: dict = {"seed": s, "noise_level": nv}
        for label, pkg in models.items():
            row[f"auc_{label}"] = evaluate_model(pkg, X_noisy, y_test, use_roc_auc=True)
        gauss_rows.append(row)

    # --- Dummy corruption sweep ---
    for flip in flip_levels:
        fv: float = round(float(flip), 2)
        X_noisy = apply_dummy_noise(X_test, fv, dummy_cols)
        row = {"seed": s, "flip_rate": fv}
        for label, pkg in models.items():
            row[f"auc_{label}"] = evaluate_model(pkg, X_noisy, y_test, use_roc_auc=True)
        dummy_rows.append(row)

    print(f"  Seed {s} done "
          f"(features: multi={sum(best_multi_ind)}, single={sum(training_results_single[s])}, "
          f"all={len(feature_names)}, forward={sum(fwd_individual)})")

gauss_df: pandas.DataFrame = pandas.DataFrame(gauss_rows)
dummy_df: pandas.DataFrame = pandas.DataFrame(dummy_rows)

# Aggregate (mean, std) across seeds
gauss_agg: pandas.DataFrame = gauss_df.drop(columns=["seed"]).groupby("noise_level").agg(["mean", "std"])
dummy_agg: pandas.DataFrame = dummy_df.drop(columns=["seed"]).groupby("flip_rate").agg(["mean", "std"])

comparison_dir: str = os.path.join(RESULT_PATH_EVAL, "all_models_comparison")
_ensure_directory(comparison_dir)
gauss_df.to_csv(os.path.join(comparison_dir, "gaussian_noise_per_seed.csv"), index=False)
dummy_df.to_csv(os.path.join(comparison_dir, "dummy_flip_per_seed.csv"), index=False)

# Plot style: one entry per model (label, color, marker, linestyle)
model_styles: dict[str, tuple] = {
    "multi":   ("Multi-Objective (SiCo-MOGA)",    "tab:blue",   "o", "-"),
    "single":  ("Single-Objective (AUC-only GA)", "tab:orange", "s", "--"),
    "all":     ("All features (no selection)",    "tab:green",  "^", "-."),
    "forward": ("Forward stepwise selection",     "tab:red",    "D", ":"),
}


def _plot_comparison(agg_df: pandas.DataFrame, x_col_name: str,
                     xlabel: str, title: str, out_path: str) -> None:
    fig, ax = plt.subplots(figsize=(10, 6))
    for key, (name, color, marker, ls) in model_styles.items():
        mean_series: pandas.Series = agg_df[(f"auc_{key}", "mean")]
        std_series: pandas.Series = agg_df[(f"auc_{key}", "std")].fillna(0.0)
        x_vals: numpy.ndarray = mean_series.index.values
        m_vals: numpy.ndarray = mean_series.values
        s_vals: numpy.ndarray = std_series.values
        ax.plot(x_vals, m_vals, label=name,
                color=color, marker=marker, linestyle=ls, linewidth=2)
        ax.fill_between(x_vals, m_vals - s_vals, m_vals + s_vals,
                        color=color, alpha=0.15)
    ax.set_xlabel(xlabel, fontweight="bold")
    ax.set_ylabel("Test ROC-AUC (mean across seeds; shaded = +/- 1 std)", fontweight="bold")
    ax.set_title(title, fontweight="bold", pad=10)
    ax.legend(frameon=True, fancybox=True, shadow=True)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


_plot_comparison(
    gauss_agg, "noise_level",
    "Gaussian Noise Level (fraction of training std, no mean shift)",
    "Robustness on Test Set: Additive Gaussian Noise on Continuous Features\n"
    "(all 4 models, averaged across seeds)",
    os.path.join(comparison_dir, "gaussian_noise_comparison_test.png"))

_plot_comparison(
    dummy_agg, "flip_rate",
    "Corruption Fraction (proportion of dummy cells randomised)",
    "Robustness on Test Set: Random Corruption Noise on Binary (Dummy) Features\n"
    "(all 4 models, averaged across seeds)",
    os.path.join(comparison_dir, "dummy_flip_comparison_test.png"))

print(f"\nAll-models comparison plots and CSVs saved to: {comparison_dir}")

## Paired Wilcoxon Signed-Rank Tests Across Seeds

For each pair of models (MOGA, SOGA, all-features, forward-stepwise) compute the **paired Wilcoxon signed-rank test** on per-seed test AUC. The pairing is by seed: at each noise level all four models are evaluated on identical data (same train/test split, same noisy `X_test` realisation), so the per-seed AUCs are matched samples — exactly what Wilcoxon assumes.

Reported quantities per pair:
- `mean_a`, `mean_b` — mean test AUC across seeds.
- `mean_diff`, `median_diff` — paired difference (a − b) summary.
- `wilcoxon_statistic`, `p_value` — two-sided test (no normality assumption).
- `significant_0.05` — convenience boolean for the conventional 0.05 cutoff.

Three tables are produced:
1. **Clean test set** (noise_level = 0.0) — the headline comparison.
2. **Across all Gaussian noise levels** — to see where significance appears/disappears under noise.
3. **Across all dummy corruption rates** — same, for the binary-feature noise sweep.

This depends on `gauss_df` and `dummy_df` produced by the *all-models comparison* cell above; run that cell first.

In [ ]:
# Paired Wilcoxon signed-rank tests across seeds (depends on gauss_df, dummy_df)
from scipy.stats import wilcoxon

# Pairs to compare (model_a, model_b) using the auc_<key> columns in gauss_df / dummy_df
wilcoxon_pairs: list[tuple[str, str]] = [
    ("multi",   "single"),
    ("multi",   "all"),
    ("multi",   "forward"),
    ("single",  "all"),
    ("single",  "forward"),
    ("all",     "forward"),
]

wilcoxon_label_map: dict[str, str] = {
    "multi":   "MOGA (SiCo)",
    "single":  "SOGA (AUC)",
    "all":     "All features",
    "forward": "Forward stepwise",
}


def _paired_wilcoxon_table(per_seed_df: pandas.DataFrame, level_col: str,
                           level_value: float) -> pandas.DataFrame:
    """Run paired Wilcoxon on every model pair at a fixed noise level.

    Returns one row per pair with means, paired-diff summary, Wilcoxon
    statistic, two-sided p-value, and a 0.05 significance flag.
    """
    sub: pandas.DataFrame = per_seed_df[
        numpy.isclose(per_seed_df[level_col], level_value)
    ]
    n_seeds: int = int(sub["seed"].nunique())
    rows: list[dict] = []
    for a, b in wilcoxon_pairs:
        x: numpy.ndarray = sub[f"auc_{a}"].to_numpy()
        y: numpy.ndarray = sub[f"auc_{b}"].to_numpy()
        diff: numpy.ndarray = x - y

        if numpy.allclose(diff, 0.0):
            stat: float = float("nan")
            p_val: float = 1.0
        else:
            stat_raw, p_raw = wilcoxon(
                x, y, zero_method="wilcox", alternative="two-sided")
            stat = float(stat_raw)
            p_val = float(p_raw)

        rows.append({
            "model_a":            wilcoxon_label_map[a],
            "model_b":            wilcoxon_label_map[b],
            "n_seeds":            n_seeds,
            "mean_a":             float(numpy.mean(x)),
            "mean_b":             float(numpy.mean(y)),
            "mean_diff":          float(numpy.mean(diff)),
            "median_diff":        float(numpy.median(diff)),
            "wilcoxon_statistic": stat,
            "p_value":            p_val,
            "significant_0.05":   bool(p_val < 0.05),
        })
    return pandas.DataFrame(rows)


# --- Headline: clean test set (Gaussian sweep at noise_level = 0.0) ---
clean_wilcoxon: pandas.DataFrame = _paired_wilcoxon_table(gauss_df, "noise_level", 0.0)

print("=== Paired Wilcoxon Signed-Rank Test: CLEAN Test AUC ===")
print(f"  n_seeds = {clean_wilcoxon['n_seeds'].iloc[0]} | alternative = two-sided\n")
print(clean_wilcoxon[[
    "model_a", "model_b", "mean_a", "mean_b",
    "mean_diff", "median_diff", "p_value", "significant_0.05",
]].to_string(index=False))

# --- Per-level: Wilcoxon at every Gaussian noise level ---
gauss_levels: list[float] = sorted(gauss_df["noise_level"].unique())
gauss_chunks: list[pandas.DataFrame] = []
for lvl in gauss_levels:
    chunk: pandas.DataFrame = _paired_wilcoxon_table(gauss_df, "noise_level", float(lvl))
    chunk["noise_level"] = float(lvl)
    gauss_chunks.append(chunk)
gauss_wilcoxon_df: pandas.DataFrame = pandas.concat(gauss_chunks, ignore_index=True)

# --- Per-level: Wilcoxon at every dummy corruption rate ---
flip_rates: list[float] = sorted(dummy_df["flip_rate"].unique())
flip_chunks: list[pandas.DataFrame] = []
for r in flip_rates:
    chunk = _paired_wilcoxon_table(dummy_df, "flip_rate", float(r))
    chunk["flip_rate"] = float(r)
    flip_chunks.append(chunk)
dummy_wilcoxon_df: pandas.DataFrame = pandas.concat(flip_chunks, ignore_index=True)

# --- Save outputs ---
wilcoxon_dir: str = os.path.join(RESULT_PATH_EVAL, "wilcoxon")
_ensure_directory(wilcoxon_dir)
clean_wilcoxon.to_csv(
    os.path.join(wilcoxon_dir, "wilcoxon_clean.csv"), index=False)
gauss_wilcoxon_df.to_csv(
    os.path.join(wilcoxon_dir, "wilcoxon_gaussian_per_level.csv"), index=False)
dummy_wilcoxon_df.to_csv(
    os.path.join(wilcoxon_dir, "wilcoxon_dummy_per_level.csv"), index=False)

# --- Quick summary: where MOGA significantly beats each baseline under noise ---
print("\n=== Where MOGA significantly differs from each baseline (Gaussian sweep) ===")
moga_rows: pandas.DataFrame = gauss_wilcoxon_df[
    gauss_wilcoxon_df["model_a"] == wilcoxon_label_map["multi"]
]
for opponent in [wilcoxon_label_map["single"], wilcoxon_label_map["all"], wilcoxon_label_map["forward"]]:
    sub = moga_rows[moga_rows["model_b"] == opponent]
    sig_levels = sub[sub["significant_0.05"]]["noise_level"].tolist()
    print(f"  MOGA vs {opponent:<18}: significant at {len(sig_levels)}/{len(sub)} noise levels "
          f"(p<0.05) -> {sig_levels}")

print(f"\nWilcoxon results saved to: {wilcoxon_dir}")